# 第4课 · 三个旋钮驯服一切声音——正弦波（sine wave）的频率、振幅（amplitude）与相位（phase，φ）

**学习目标**
1. 掌握 `A · sin(2πft + φ)` 的三个参数各控制什么
2. 实现 `sinusoid(t, A, f, phi)` 并用 assert 验证
3. 理解频率叠加：和弦（chord） = 多个正弦波相加
4. 为 L05（复数，complex number）和 L06（欧拉公式，Euler's formula）打好基础

**Aurora 连接**：`aurora.audio.sine`（`src/aurora/audio/io.py`）用同一公式生成正弦波，但接口不同（无 `t` 数组、无 `phi` 参数）；
L37 DFT 把任意信号分解成正弦波之和——分解的「原子」就是本节的 `sinusoid`。

**来自 L03 的记忆**：谱图（spectrogram）上的每条水平亮线，对应的就是一个特定频率的正弦波。
本课让你亲手控制那条线的位置（频率）、亮度（振幅）和起始角度（相位）。

← **上一课**　[L03 · 谱图直觉](../0_foundation/L03_spectrogram.ipynb)

> 上节课学习了 **谱图直觉**：在学 FFT 之前先读懂时频图的三个轴。  
> 本课将探讨 **正弦波三要素**。

## 本课剧情：三个旋钮控制一切

想象一个收音机调台器。它有三个旋钮：

- **音量旋钮**（振幅 A）：拧大 → 声音变响；拧小 → 变轻。范围从 0 到正无穷。
- **频率旋钮**（频率 f）：拧大 → 声调变高（像女高音）；拧小 → 变低（像低音炮）。
- **时间偏移旋钮**（相位 φ）：整段声音往前或往后移动一点，起点不再是波峰，而是中间的某个位置。

一个正弦波完全由这三个旋钮决定：

| 旋钮 | 符号 | 物理意义 | 谱图上的效果 |
|---|---|---|---|
| 音量 | A（振幅） | 多响 → 波峰高度 | 颜色更亮 |
| 频率 | f（Hz）| 多快 → 单位时间几个周期 | 亮线位置更高 |
| 相位 | φ（rad）| 起点在哪 → 波从哪里开始 | 谱图不变，波形水平移 |

公式：`x(t) = A · sin(2π·f·t + φ)`

本课你将实现 `sinusoid(t, A, f, phi)`——整个 Aurora DSP 管线的基础构件之一。

## 1. sin 与 cos：同一个圆的两个侧面

有没有想过：为什么 sin 和 cos 长得这么像，却不完全一样？

把一个点放在单位圆上，让它匀速逆时针转。此刻：
- **cos(θ)** = 它在 x 轴（水平方向）上的"影子"
- **sin(θ)** = 它在 y 轴（垂直方向）上的"影子"

cos 超前 sin 恰好 π/2（90°）。也就是说：

```
cos(θ) = sin(θ + π/2)
```

两个函数，同一个旋转，不同的投影方向。后面复数（L05）会把这两个影子打包成一个数——那时候你会发现 Euler 公式 `e^{iθ} = cos θ + i·sin θ` 正好描述了这个旋转点。

## 实验入口：角度、坐标和旋转

sin 和 cos 描述同一个圆周运动的两个分量：cos 比 sin 超前四分之一个周期。观察两条曲线的峰值、零点、谷值的位置关系，确认 cos(0)=1 而 sin(0)=0。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 2*np.pi, 400)
plt.plot(t, np.sin(t), label='sin')
plt.plot(t, np.cos(t), label='cos')
plt.legend(); plt.title('sin & cos'); plt.grid(True, alpha=.3); plt.show()

## 动手观察：四个特殊角度

0、π/2、π、3π/2 这四个角度的 cos 和 sin 值都是 0 或 ±1，可以直接心算验证。运行后比对 `z.real` 和 `z.imag`，确认输出与预期完全一致。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：遍历频率、振幅和相位

采样率（sampling rate / sample rate，sr） 16 Hz、时长 0.5 s 只产生 8 个采样点，数字少到可以逐个检查：freq 加倍时相邻样本的最大变化幅度明显增大（f=1→2 约 1.85 倍，f=2→4 约 1.41 倍——正弦是弯的，并非精确翻倍），amplitude 加倍时输出的 min/max 同比例放大。

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## 2. 一个正弦波 = `A · sin(2π·f·t + φ)`

- **A 振幅** = 多响（把 `[-1, 1]` 的输出拉伸到 `[-A, A]`）
- **f 频率** = 多高，Hz（每秒完整振荡次数，决定音高）
- **φ 相位** = 起始角度偏移，单位弧度（φ=π/2 时波形从峰值开始）

三个参数互相正交：A 只缩放幅度、f 只改变时间轴上的振荡速率、φ 只偏移起始角度——改任何一个不影响另外两个。这样在合成复杂波形（如和弦、MFCC filterbank 的各个分量）时，每个成分都能单独调，不用回头再去动另外两个。

## 3. ✏️ 实现 `sinusoid(t, A, f, phi)`

**推理路线**：
1. `t` 是时间数组（秒），`f` 是每秒完整周期数（Hz）。一个完整周期对应 `2π` 弧度，所以时刻 `t` 处的角度是 `2π·f·t`。
2. 加上初相位（initial phase） `phi`（弧度），得到完整角度序列 `2π·f·t + phi`。
3. `np.sin(...)` 把角度映射到 `[-1, 1]`；乘以 `A` 把值域扩展到 `[-A, A]`。注意顺序：先算角度，再一次性乘 A，避免多次标量乘法。

**参考输入输出**：`A=2, f=1, phi=0, t=[0, 0.25, 0.5, 0.75]` → `[0, 2, 0, -2]`（一个周期的四个采样点）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `sinusoid` 前明确三件事：
- 输入：`t`（时间数组）、`A`（振幅）、`f`（频率 Hz）、`phi`（初相位，默认 0）
- 关键步骤：计算相位序列 `2π·f·t + phi`，乘以 A
- 返回：与 `t` 同形状的浮点数组，值域在 `[-A, A]`

In [ ]:
def sinusoid(t, A, f, phi=0.0):
    # ✏️ TODO: 返回正弦波
    raise NotImplementedError("TODO: 实现 sinusoid(t, A, f, phi)")


In [ ]:
import numpy as np
t = np.linspace(0, 1, 8, endpoint=False)
try:
    y = sinusoid(t, 2.0, 1.0, 0.0)
    assert np.allclose(y, 2*np.sin(2*np.pi*1.0*t)), '应等于 2·sin(2πt)'
    # 额外检验：phi=π/2 时应等于 cos
    y_cos = sinusoid(t, 1.0, 1.0, np.pi/2)
    assert np.allclose(y_cos, np.cos(2*np.pi*t), atol=1e-10), 'phi=π/2 时应等于 cos'
    print('✅ 通过：你能造任意振幅/频率/相位的正弦波了。')
except (NotImplementedError, TypeError):
    print('⚠️ sinusoid() 还未实现，请先完成上方 TODO 再运行本格。')


**🔗 Aurora 连接**：`aurora.audio.sine`（见 `src/aurora/audio/io.py`）采用同一个公式，但接口不同——它接受 `freq, duration, sample_rate` 而非时间数组 `t`，且不含 `phi` 参数。你将在 L33 实现完全对齐的 `my_sine`。下一课进入复数——FFT 的输出类型。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | 过零次数≈{zero_crossings}')


## 参数实验：只改一个旋钮

把 `f` 从 1 改到 4，观察相同 `t` 范围内过零（zero crossing）次数从 1 次涨到 7 次——规律约为 2f−1（f=1,2,4,8 对应 1,3,7,15），频率翻倍时过零次数大约翻倍，并不是简单的 4 倍正比。再把 `phi` 改到 `np.pi/2`，确认波形从余弦形状（先到峰值）开始。

## 参数实验：和弦 = 多个正弦波相加

任何复杂声音都可以分解成正弦波之和（这正是 Fourier 分析的核心思想）。
反过来，把多个不同频率的 `sinusoid` 加在一起，就能合成和弦。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr

try:
    # C 大调和弦：C4 + E4 + G4
    c4 = sinusoid(t, A=1.0, f=261.6)
    e4 = sinusoid(t, A=0.8, f=329.6)
    g4 = sinusoid(t, A=0.6, f=392.0)
    chord = (c4 + e4 + g4) / 3.0  # 除以 3 防止三个振幅叠加后超出 [-1,1] 范围

    fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
    for ax, sig, label in zip(axes,
            [c4, e4, g4, chord],
            ['C4 (261.6 Hz)', 'E4 (329.6 Hz)', 'G4 (392.0 Hz)', '和弦（三者叠加）']):
        ax.plot(t, sig, lw=0.8)
        ax.set_ylabel(label, fontsize=8); ax.set_ylim(-1.4, 1.4); ax.grid(alpha=0.3)
    axes[-1].set_xlabel('时间 (s)')
    fig.suptitle('三个正弦波 → 叠加成和弦')
    plt.tight_layout(); plt.show()
    print('DFT 的任务就是从「和弦」这条曲线，分解回 C4/E4/G4 三条。')
except (NotImplementedError, TypeError):
    print('⚠️ 先完成上方 sinusoid() 练习再运行本格（和弦演示需要你的实现）。')


In [ ]:
# 小检查：同一个角度，同时看 cos、sin、复数
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')


## 本课收束

现在你能用 `sinusoid(t, A, f, phi)` 生成任意振幅、频率、相位的正弦波，
并用 `assert np.allclose` 验证数值是否正确。

**和弦 = 正弦波叠加**：两个不同频率的 `sinusoid` 相加，谱图上就出现两条亮线——
这正是 DFT 反问题的直觉：「给定叠加的波形，分解出每条线的位置和高度」。

**下一课 L05**：复数的模与相位——给每个频率发一张身份证，读出模与相位，这是 FFT 的母语。
FFT 的每个输出频点（frequency bin）都是一个复数：**模 = 该频率有多响，相位 = 时间对齐信息**。
今天的 A 和 φ，就对应那个复数的模和相位。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：三要素手算（目标 8 分钟）

盖上屏幕，纸上回答：

**参数**：A = 2，f = 440 Hz，φ = π/4，sr = 8000 Hz

**问 1**：写出 `x(t)` 的完整公式（用数字代入，不简化）。

**问 2**：`t = 1/880` 秒时，x 的精确值是多少？
提示：`sin(2π·440·(1/880) + π/4)` = `sin(π + π/4)` = ?

**问 3**：频率 440 Hz、采样率 8000 Hz，一个周期内有多少个采样点？（不必是整数）

**问 4**：若把 φ 改为 0，x(0) 的值变成多少？（原来 φ=π/4 时 x(0) 是多少？）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

A, f, phi, sr = 2.0, 440.0, np.pi/4, 8000

# 问1：公式验证（用代码验证你的公式）
t_test = np.array([0.0, 1/880, 1/440])
x_formula = A * np.sin(2 * np.pi * f * t_test + phi)
print(f"Q1 ✅  x(t) = {A} · sin(2π·{f:.0f}·t + π/4)")
print(f"       x(0)      = {x_formula[0]:.4f}")
print(f"       x(1/880)  = {x_formula[1]:.4f}")

# 问2：手算验证
t2 = 1/880
theta = 2 * np.pi * f * t2 + phi
# 2π·440·(1/880) = 2π·0.5 = π，所以 θ = π + π/4 = 5π/4
x2_theory = 2 * np.sin(5 * np.pi / 4)  # = 2·(-√2/2) = -√2
x2_code   = A * np.sin(2 * np.pi * f * t2 + phi)
assert np.isclose(x2_code, x2_theory, atol=1e-10), f"Q2 差异: {x2_code:.6f} vs {x2_theory:.6f}"
print(f"\nQ2 ✅  x(1/880) = 2·sin(5π/4) = 2·(-√2/2) = -√2 ≈ {x2_theory:.4f}  代码值 {x2_code:.4f}")

# 问3：每周期采样点数
samples_per_cycle = sr / f
print(f"\nQ3 ✅  每周期采样点 = {sr}/{f:.0f} = {samples_per_cycle:.4f} 点/周期")

# 问4：相位为0时 x(0)
x_phi0 = A * np.sin(2 * np.pi * f * 0 + 0)    # = 0
x_phiQ = A * np.sin(2 * np.pi * f * 0 + phi)  # = 2·sin(π/4) = √2
assert np.isclose(x_phi0, 0.0, atol=1e-12)
print(f"\nQ4 ✅  φ=0 时 x(0) = {x_phi0:.4f}  φ=π/4 时 x(0) = {x_phiQ:.4f} (= √2 ≈ {np.sqrt(2):.4f})")
print("\n🎉 白板挑战通过！三要素 A / f / φ 的手算已经内化。")

In [ ]:
# ✏️ 本课自评
l04_review = {
    "sinusoid_implemented":   None,  # sinusoid 函数实现并通过断言？True/False
    "three_params_understood": None, # 能解释 A/f/φ 各自对波形的影响？True/False
    "whiteboard_passed":       None, # 白板挑战纸上推导完成？True/False
    "chord_intuition":         None, # 理解"和弦 = 多正弦波叠加"？True/False
}

unfilled = [k for k, v in l04_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l04_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L04 全部通关！进入 L05：复数几何本质')

---

→ **下一课**　[L05 · 复数的模与相位](L05_complex_numbers.ipynb)

> 下节课将学习 **复数的模与相位**：给每个频率发一张身份证——从复数读出模与相位，这是 FFT 的母语。